# Atelier 1 — Cadrer Copilot dans Fabric

**Cas métier :** Maison Lumière · **Corrigé :** observer, guider et vérifier Copilot sur 60 000 lignes de ventes.

- Attacher un Lakehouse par défaut.
- Charger les quatre CSV dans `Files/atelier1/`.
- Exécuter les cellules dans l'ordre.
- Les cellules Copilot restent des expériences : les chiffres ci-dessous viennent du générateur à graine fixe.
- Fabric fournit le terrain ; l'objet évalué est la manière d'utiliser Copilot.

## Partie 0 — Activer et configurer Copilot

**Dictionnaire :** ouvrir `data input/dictionnaire_donnees.md` dans un second onglet ou dans l'éditeur des supports ; ne pas l'importer dans une cellule du notebook.

**Copilot :** dans le notebook ouvert, sélectionner **Copilot** dans la barre de commandes située au-dessus des cellules. Le volet doit s'ouvrir à droite.

**Retour au notebook :** après l'observation des paramètres du locataire dans OneLake catalog, revenir dans ce notebook. Réaliser dans le volet Copilot les essais **Ask/Don't ask**, puis **Keep**, **Undo** et la vue des différences sur une modification inoffensive de `BASE_PATH`. Revenir ensuite à **Ask**.

**Gouvernance :** dans **OneLake catalog > Govern > Configurations > Tenant settings**, utiliser **Filter by keyword** avec le libellé exact `Users can use Copilot and other features powered by Azure OpenAI`. Ouvrir le résultat de la catégorie **Copilot and Azure OpenAI Service**, puis relever son état et son périmètre sans les modifier.

**Surfaces à repérer :** distinguer le volet **Copilot** du notebook, Copilot au-dessus d'une cellule avec `/explain`, `/fix`, `/comments` et `/optimize`, le menu **AI tools** `[À VÉRIFIER SUR TENANT]` et **Open In VS Code (Desktop)**. Dans VS Code, l'assistant est GitHub Copilot Chat avec le contexte fourni par l'extension Fabric Data Engineering.

**Repères de navigation :** OneLake catalog sert uniquement à l'observation de la gouvernance ; le Lakehouse sert au dépôt ou au renommage des fichiers ; les interactions principales se font dans le notebook. Data Wrangler s'ouvre depuis le notebook et se ferme pour y revenir.

## Partie A — Ce que Copilot reçoit comme contexte

**Avant la cellule suivante :** interroger Copilot par tours courts : capacité oui/non, inventaire des fichiers, colonnes en JSON, puis types cibles estimés pour un seul fichier. Ne cumuler ni calcul exact ni transformation dans ces demandes.

**Après la cellule suivante :** confronter les types estimés à la sortie `Types bruts`; les colonnes sont chargées comme texte.

**Blocage :** refuser toute cellule proposée pendant l'inventaire. Si le volet ou le notebook se fige, ouvrir la même URL dans un nouvel onglet, vérifier le notebook, puis fermer l'ancien onglet. **Stop session** ne réinitialise pas Copilot.

In [ ]:
import pandas as pd

BASE_PATH = "Files/atelier1"

def read_csv_as_text(filename):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{BASE_PATH}/{filename}")
        .toPandas()
    )

df_ventes_raw = read_csv_as_text("ventes_brut.csv")
df_magasins = read_csv_as_text("magasins.csv")
df_produits = read_csv_as_text("produits.csv")
df_retours = read_csv_as_text("retours.csv")

print("Ventes :", df_ventes_raw.shape)
print("Types bruts :", df_ventes_raw.dtypes.astype(str).to_dict())
print("Magasins :", df_magasins.shape)
print("Produits :", df_produits.shape)
print("Retours :", df_retours.shape)

In [ ]:
profil_brut = {
    "lignes": len(df_ventes_raw),
    "colonnes": len(df_ventes_raw.columns),
    "doublons_exacts": int(df_ventes_raw.duplicated().sum()),
    "dates_avec_slash": int(df_ventes_raw["date_vente"].str.contains("/", regex=False).sum()),
    "montants_avec_virgule": int(df_ventes_raw["montant"].str.contains(",", regex=False).sum()),
    "montants_avec_point": int(df_ventes_raw["montant"].str.contains(".", regex=False).sum()),
}
profil_brut

**Attendu :** 60 000 lignes, 8 colonnes, 500 doublons exacts, 24 722 dates avec `/`, 20 179 montants avec virgule et 45 872 avec point. Les deux derniers groupes se chevauchent lorsque les deux séparateurs sont présents.

### A.5 — Initialiser puis renseigner le journal

**Première utilisation :** créer maintenant `Journal_Copilot`, renseigner `0_activation` et `A_contexte`, puis conserver le même dictionnaire pour B.6, C.6, D.6 et E.11. Ne pas réexécuter l'initialisation après les mises à jour suivantes.

In [ ]:
# À exécuter une seule fois en A.5 : B.6, C.6, D.6 et E.11 mettent ensuite ces clés à jour.
# Une nouvelle exécution de cette cellule efface les observations déjà saisies.
Journal_Copilot = {
    "0_activation": "<observation>",
    "A_contexte": "<observation>",
    "B_prompts": "<observation>",
    "C_interfaces": "<observation>",
    "D_conversation": "<observation>",
    "E_relecture": "<observation>",
}
Journal_Copilot

## Partie B — Itérer sur le prompt

**Itération :** comparer « Nettoie les ventes », un prompt avec noms et contrôles, puis un prompt ancré dans le dictionnaire métier. Après un résultat faux, reformuler en citant l'écart numérique observé.

## Partie C — Choisir le mode d'interaction

**Dans la cellule :** utiliser `/explain`, `/fix`, `/comments` ou `/optimize` sur une cellule ciblée.

**Complétion en ligne :** commencer `nb_doublons =` après un commentaire explicite, puis valider que le résultat vaut 500.

**Diagnostic SparkR :** provoquer dans l'atelier une erreur avec deux directives `%%sparkr`, sélectionner **Diagnose with Copilot** — **Fix with Copilot** dans Microsoft Learn — puis ne conserver qu'une directive après examen du diff. La cellule corrigée se trouve juste après ce repère.

**AI tools :** inventorier les entrées du menu sans lancer de traitement et distinguer ces outils du volet Copilot.

**VS Code :** avec l'extension Fabric Data Engineering déjà installée, ouvrir **Open In VS Code (Desktop)**, puis utiliser GitHub Copilot Chat avec l'agent par défaut et une session **Local** pour expliquer une seule cellule sans la modifier.

**Compatibilité :** si `%%code` ou `%%chat` sont proposés dans l'environnement, comparer leur comportement au panneau ; sinon consigner leur absence sans bloquer l'atelier.

In [ ]:
%%sparkr

data <- data.frame(
  nom = c("Alice", "Bob", "Charlie"),
  age = c(30, 25, 35)
)

df_exemple_r <- createDataFrame(data)
display(df_exemple_r)

## Partie D — Suivre une conversation

**Contrat initial :** demander une fonction `preparer_ventes_conversation(df)` qui conserve 59 500 lignes, convertit tous les montants, retrouve 111 350 063,34 EUR et limite les dates du 1er janvier au 31 mars 2026.

**Même conversation, six tours :** dédoublonnage, montants, dates, assertions, commentaire d'une règle, puis optimisation sans changement de résultat. Relever tout oubli du contrat ou dérive de nom.

**Correction :** signaler l'écart dans la même conversation. Rejouer ensuite le contrat dans une nouvelle conversation et comparer la continuité, la concision et les contrôles conservés.

In [ ]:
def mesurer_proposition(df_prepare):
    return {
        "lignes": len(df_prepare),
        "montants_nuls": int(df_prepare["montant_num"].isna().sum()),
        "somme_montant": round(df_prepare["montant_num"].sum(), 2),
        "date_min": str(df_prepare["date"].min().date()),
        "date_max": str(df_prepare["date"].max().date()),
    }

# Après génération par Copilot :
# mesurer_proposition(preparer_ventes_conversation(df_ventes_raw))

## Partie E — Relire avant d'exécuter

**Data Wrangler, témoin court :** comparer l'échantillon des 5 000 premières lignes à un échantillon aléatoire ; Data Wrangler n'est pas une fonctionnalité Copilot.

**Décision :** inspecter le diff, exécuter des contrôles de volume, somme et cardinalité, puis choisir **Keep** ou **Undo**. Les cellules suivantes rendent reproductibles trois propositions plausibles mais fausses.

In [ ]:
df_test = df_ventes_raw.drop_duplicates(subset=["id_ligne"], keep="first").copy()

montant_naif = pd.to_numeric(
    df_test["montant"].str.replace(",", ".", regex=False),
    errors="coerce",
)

print("Lignes métier :", len(df_test))
print("Montants convertis :", int(montant_naif.notna().sum()))
print("Montants devenus nuls :", int(montant_naif.isna().sum()))
print("Somme naïve :", round(montant_naif.sum(), 2))

**Attendu :** 59 500 lignes, 54 842 montants convertis, 4 658 nulls et une somme naïve de 94 478 059,58 EUR.

In [ ]:
def parse_montant(valeur):
    texte = str(valeur).strip().replace(" ", "")
    if "," in texte and "." in texte:
        if texte.rfind(",") > texte.rfind("."):
            texte = texte.replace(".", "").replace(",", ".")
        else:
            texte = texte.replace(",", "")
    elif "," in texte:
        texte = texte.replace(",", ".")
    return float(texte)

montant_corrige = df_test["montant"].map(parse_montant)
print("Montants convertis :", int(montant_corrige.notna().sum()))
print("Montants devenus nuls :", int(montant_corrige.isna().sum()))
print("Somme corrigée :", round(montant_corrige.sum(), 2))

**Attendu :** 59 500 montants convertis, aucun null et 111 350 063,34 EUR.

In [ ]:
date_naive = pd.to_datetime(
    df_test["date_vente"], format="mixed", dayfirst=False, errors="raise"
)

mask_iso = df_test["date_vente"].str.match(r"^\d{4}-\d{2}-\d{2}$")
date_corrigee = pd.Series(pd.NaT, index=df_test.index, dtype="datetime64[ns]")
date_corrigee.loc[mask_iso] = pd.to_datetime(
    df_test.loc[mask_iso, "date_vente"], format="%Y-%m-%d", errors="raise"
)
date_corrigee.loc[~mask_iso] = pd.to_datetime(
    df_test.loc[~mask_iso, "date_vente"], format="%d/%m/%Y", errors="raise"
)

print("Dates différentes :", int((date_naive != date_corrigee).sum()))
print(date_corrigee.dt.to_period("M").value_counts().sort_index())
print(date_naive.dt.to_period("M").value_counts().sort_index())

**Attendu :** 9 042 dates différentes. Le parseur corrigé contient 20 433 lignes en janvier, 18 523 en février et 20 544 en mars ; le parseur naïf crée aussi des mois d'avril à décembre.

In [ ]:
df_test["montant_num"] = montant_corrige
df_test["date"] = date_corrigee

jointure_magasin_fausse = df_test.merge(
    df_magasins, left_on="id_magasin", right_on="store_code", how="left"
)

print("Lignes avant :", len(df_test))
print("Lignes après :", len(jointure_magasin_fausse))
print("Somme avant :", round(df_test["montant_num"].sum(), 2))
print("Somme après :", round(jointure_magasin_fausse["montant_num"].sum(), 2))
print(df_magasins["store_code"].value_counts().head())

**Attendu :** 59 500 → 64 510 lignes et 111 350 063,34 → 120 662 262,79 EUR ; `MAG-004` apparaît deux fois.

In [ ]:
df_magasins_clean = df_magasins.loc[
    ~df_magasins["nom_magasin"].str.contains("ancienne fiche", case=False, na=False)
].copy()
assert not df_magasins_clean["store_code"].duplicated().any()

jointure_magasin_ok = df_test.merge(
    df_magasins_clean,
    left_on="id_magasin",
    right_on="store_code",
    how="left",
    validate="many_to_one",
)
print(len(jointure_magasin_ok), round(jointure_magasin_ok["montant_num"].sum(), 2))

In [ ]:
retours_faux = df_test.merge(
    df_retours, on="id_vente", how="inner", suffixes=("_vente", "_retour")
)
retours_ok = df_test.merge(
    df_retours, on=["id_vente", "id_produit"], how="inner", validate="one_to_one"
)
produit_identique = (
    retours_faux["id_produit_vente"] == retours_faux["id_produit_retour"]
)
print("Jointure sur id_vente :", len(retours_faux))
print("Produits réellement identiques :", int(produit_identique.sum()))
print("Clé composée :", len(retours_ok))

**Attendu :** 9 000 lignes sur `id_vente` seul, dont 3 000 produits identiques ; 3 000 lignes avec la clé composée.

In [ ]:
controle_relecture = {
    "lignes_metier": len(df_test),
    "somme_corrigee": round(montant_corrige.sum(), 2),
    "ecart_dates": int((date_naive != date_corrigee).sum()),
    "lignes_jointure_fausse": len(jointure_magasin_fausse),
    "lignes_jointure_corrigee": len(jointure_magasin_ok),
    "retours_cle_incomplete": len(retours_faux),
    "retours_cle_composee": len(retours_ok),
}
controle_relecture

In [ ]:
assert controle_relecture == {
    "lignes_metier": 59_500,
    "somme_corrigee": 111_350_063.34,
    "ecart_dates": 9_042,
    "lignes_jointure_fausse": 64_510,
    "lignes_jointure_corrigee": 59_500,
    "retours_cle_incomplete": 9_000,
    "retours_cle_composee": 3_000,
}
print("Contrôles de relecture validés")

**Attendu :** chaque proposition est acceptée ou rejetée à partir d'un écart mesuré, pas de sa seule plausibilité.

In [ ]:
Journal_Copilot["E_relecture"] = (
    "Une proposition plausible a été comparée à des contrôles de volume, "
    "de somme et de cardinalité avant Keep ou Undo."
)
Journal_Copilot

In [ ]:
Journal_Copilot["0_activation"] = "<Ask/Don't ask, diff, Keep/Undo, gouvernance>"
Journal_Copilot["A_contexte"] = "<schéma, échantillon, sorties, source absente>"
Journal_Copilot["B_prompts"] = "<vague, précis, ancré, reformulé>"
Journal_Copilot["C_interfaces"] = "<panneau, cellule, complétion, diagnostic SparkR, compatibilité>"
Journal_Copilot["D_conversation"] = "<même conversation et nouvelle conversation>"
Journal_Copilot

## Synthèse — Relire le `Journal_Copilot`

**À compléter :** relire les six entrées, puis rédiger trois ou quatre phrases libres sur le contexte reçu, l'itération des prompts, la continuité de la conversation et la relecture avant exécution.

**Trace :** relier les observations aux nombres exécutés dans ce notebook ; aucune liste de conseils finale n'est fournie.

In [ ]:
Journal_Copilot